# Tutorial 1: SQD-GQE on the ffsim simulator (train + inference)

This notebook walks through the full **SQD-GQE** loop on the `ffsim` statevector simulator: a transformer policy proposes short quantum circuits, `ffsim` samples exact Born-rule bitstrings from them, and `qiskit-addon-sqd` diagonalizes the sampled subspace to get an energy. It covers both **training** (learning the policy) and **inference** (using a trained checkpoint zero-shot).

Install: `pip install -e '.[sqd]'` (see `tutorial/03_environment_setup.md` for full environment setup, including multi-GPU).

For how real IBM Quantum hardware plugs into the exact same loop instead of `ffsim`, see `tutorial/02_ibm_hardware.ipynb`.

## 1. Build the molecular system
`MolecularSystem` runs (RO)HF, builds the CAS integrals, and provides CASCI/CCSD references. We use H2 (STO-3G, 2 electrons / 2 orbitals = 4 qubits) here so every cell runs in seconds -- swap in any `molecules/*.yaml` for a real molecule.

In [1]:
from omegaconf import OmegaConf
from sqd_gqe.core.system import MolecularSystem

geometry = OmegaConf.create({"type": "linear_chain", "atoms": ["H", "H"], "bond_length": 0.74})
system = MolecularSystem(geometry=geometry, basis="sto-3g", nelecas=2, norbcas=2, spin=0, charge=0)
print("norb", system.norb, "nelec", system.nelec, "qubits", system.norb * 2)
print("CASCI", system.casci_energy(), "CCSD", system.ccsd_energy())

converged SCF energy = -1.11675930739643


norb 2 nelec (1, 1) qubits 4
CASCI -1.137283834488502 CCSD -1.1372839986104384


<repo>/.venv_local/lib/python3.10/site-packages/pyscf/lib/misc.py:149: UserWarning: OpenMP is not available. Setting omp_threads to 1 has no effects.
  warnings.warn('OpenMP is not available. '


## 2. Build the ffsim operator pool
The pool mixes a CCSD-seeded LUCJ/UCJ layer (the chemistry inductive bias) with hardware-efficient local primitives (orbital rotations, Givens rotations, diagonal-Coulomb evolutions, number-number interactions). Each entry is one particle-number-conserving `ffsim` unitary; `pool[0]` is the identity. This is the fixed "vocabulary" the transformer policy samples from -- see the architecture deck in `docs/2026-08-21/` for the full picture.

**The local primitives' angles are CCSD-derived too, not fixed constants.** `FfsimLUCJPool`'s `angle_source` picks how:
- `"ffsim_native"` (**default**) -- derives the angle scale from the same real ffsim CCSD-amplitude factorization the LUCJ op itself uses internally (`ffsim.linalg.double_factorized_t2` + `orbital_rotation_from_t1_amplitudes`). Best measured accuracy of the two options.
- `"rms"` -- a simpler heuristic: the RMS magnitude of the raw t1/t2 amplitude tensors. Measured to give more frequent (but noisier) training-gate-accepted updates than `"ffsim_native"`.

Measured on N2 (16 qubits), 20 training iterations, counting only gate-accepted iterations:

| `angle_source` | gate PASS rate | mean \|err\| (mHa) | max \|err\| (mHa) |
|---|---|---|---|
| fixed `[pi/4, pi/2]` (old default, pre-fix) | 40% | 0.0147 | 0.0756 |
| `"rms"` | **65%** | 0.0390 | 0.3230 |
| `"ffsim_native"` (current default) | 55% | **0.0114** | **0.0630** |

Switch it via config (`operator_pool.angle_source: "rms"` in `configs_sqd/operator_pool/ffsim_lucj.yaml`), a Hydra CLI override (`--hydra-override operator_pool.angle_source=rms`), or directly in Python as shown below. See `FfsimLUCJPool._ccsd_scaled_angles`'s docstring in `sqd_gqe/gqe/operator_pool.py` for exactly how to reproduce the table above.

In [2]:
from sqd_gqe.gqe.operator_pool import FfsimLUCJPool

pool = FfsimLUCJPool(system)
print("vocab size", pool.get_vocab_size())
for op in list(pool)[:6]:
    print(op.name, dict(op.gate_count))

vocab size 11
I {}
LUCJ(reps=2) {'cx': 14, 'total': 28}
orb_rot[0] {'cx': 2, 'total': 4}
orb_rot[1] {'cx': 2, 'total': 4}
orb_rot[2] {'cx': 2, 'total': 4}
orb_rot[3] {'cx': 2, 'total': 4}


In [3]:
# Compare the two CCSD-derived angle sources directly (params=None picks up
# angle_source; passing an explicit params list, as in the earlier fixed-angle
# baseline, always overrides angle_source outright).
pool_native = FfsimLUCJPool(system, angle_source="ffsim_native")  # default
pool_rms = FfsimLUCJPool(system, angle_source="rms")
print("ffsim_native angles (rad):", pool_native._ccsd_scaled_angles())
print("rms angles (rad):        ", pool_rms._ccsd_scaled_angles())

ffsim_native angles (rad): [0.056632191612346304, 0.1698965748370389]
rms angles (rad):         [0.04004500672254599, 0.12013502016763797]


## 3. Sample determinants and diagonalize with SQD (one manual pass)
Apply a short random operator sequence, sample bitstrings from the exact `ffsim` statevector, and diagonalize the sampled subspace. This is the same `SqdPipeline.run()` call that training's reward and inference both go through -- everything downstream from here is identical whether the circuit came from `ffsim` or real hardware (see `tutorial/02_ibm_hardware.ipynb`).

In [4]:
import numpy as np, torch
from sqd_gqe.gqe.sampler import FfsimSampler
from sqd_gqe.sqd.pipeline import SqdPipeline

sampler = FfsimSampler(pool, shots_count=50_000, seed=0)
pipeline = SqdPipeline(system, pool, sampler, max_size=2000, diag_backend="sqd", spin_sq=0.0)

rng = np.random.default_rng(0)
seq = torch.tensor([[0] + list(rng.integers(0, len(pool), size=8)) for _ in range(4)])
report = pipeline.run({"idx": seq})
print("sample energies", report.energies)
print("local refined", report.local_refined.energy)
print("pipeline timings", pipeline.timings)

sample energies [-1.1372838344885017, -1.1372838344885017, -1.1372838344885017, -1.1372838344885017]
local refined -1.1372838344885017
pipeline timings {'sample': 0.05711470800451934, 'diagonalize': 0.006002834008540958, 'refine_local': 0.0008808749844320118}


## 4. Training: the real GQE loop
The full loop -- policy samples a circuit batch, `ffsim` samples bitstrings, SQD diagonalizes for an energy, GRPO turns that into a loss, AdamW updates the policy -- is driven by `benchmark/amortized_train.py`. This is the exact script used for every real training run in this project (including on real IBM hardware -- see notebook 2).

Here we run just **2 iterations** on H2 so it finishes in seconds; a real molecule/run uses far more (`--iters 30` was typical, taking ~15-25 min/iteration once GPU-based SQD recovery is enabled -- see `docs/2026-08-21/pipeline_overview_2026-08-21.pdf` for real timing numbers).

In [5]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "../benchmark/amortized_train.py",
        "--molecules", "../molecules/small/h2.yaml",
        "--backend", "ideal",
        "--iters", "2",
        "--ngates", "4",
        "--shots", "2000",
        "--max-dim", "500",
        "--out", "/tmp/tutorial_h2_run.json",
        "--ckpt", "/tmp/tutorial_h2_policy.pt",
        "--checkpoint-every", "100",
    ],
    capture_output=True, text=True, cwd=".",
)
print(result.stdout[-2500:])
if result.returncode != 0:
    print(result.stderr[-2500:])

converged SCF energy = -1.11675930739643
Vocab size: 32
[AMORTIZED] it=  0 mol=h2     loss=3.118 err=+0.000mHa depth=10 2q=8 dim=4 grad=7.575 dW=0.0199 beta=0.520 gate=PASS wall=2.2s [sample=0.0s diag=0.0s refine=0.0s transpile=0.0s backward=2.1s]
  -> new best checkpoint (chem_acc=True, score=10) saved to /tmp/tutorial_h2_policy_best.pt
[AMORTIZED] it=  1 mol=h2     loss=2.884 err=+0.000mHa depth=11 2q=8 dim=4 grad=7.760 dW=0.0208 beta=0.540 gate=PASS wall=2.0s [sample=0.0s diag=0.0s refine=0.0s transpile=0.0s backward=1.9s]
saved checkpoint -> /tmp/tutorial_h2_policy.pt
wrote <repo>/experiments/amortized/amortized_diagnostics.png
wrote <repo>/experiments/amortized/depth_energy_vs_iter.png



## 5. Inference: zero-shot from a trained checkpoint
`sqd_gqe.run.run_molecule` loads a checkpoint and runs the trained policy zero-shot -- no further training -- on a molecule (the same or a different one). This is the exact function used for every zero-shot inference result in this project, including the **+0.0046 mHa (exact chemical accuracy)** 32-qubit result described in `docs/2026-08-21/pipeline_overview_2026-08-21.pdf`.

`--max-dim` controls the SQD subspace size at inference time -- independent of whatever subspace size training used, and often the single biggest lever for accuracy (see that same deck's "32 qubits: exact chemical accuracy" slide).

In [6]:
from sqd_gqe.run import run_molecule

report = run_molecule(
    "../molecules/small/h2.yaml",
    backend="ideal",
    checkpoint="/tmp/tutorial_h2_policy_best.pt",
    adapt_iters=5,
    max_dim=500,
)
print(report.summary())

[run] building system...


converged SCF energy = -1.11675930739643


<repo>/.venv_local/lib/python3.10/site-packages/pyscf/lib/misc.py:149: UserWarning: OpenMP is not available. Setting omp_threads to 1 has no effects.
  warnings.warn('OpenMP is not available. '
[run] system built: norb=2 nelec=(1, 1)


Vocab size: 32


<repo>/.venv_local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31), got 50256. This may result in unexpected behavior.


... (13 outputs left out)

[run] computing circuit depth/2q-gate metrics on the target device...


[run] circuit metrics: depth=45 2q_gates=40 within_budget=True


[run] final: energy=-1.137284 dim_used=4 shots_used=100000 -- assembling report...


=== SQD-GQE chemistry report: h2 on ideal ===
  qubits=4  electrons=2  charge=0  2S=0  (zero-shot)

  Total energy (SQD-GQE) : -1.13728383 Ha
  CASCI (exact)          : -1.13728383 Ha
  CCSD                   : -1.13728400 Ha
  Hartree-Fock           : -1.11675931 Ha
  Error vs CASCI         : +0.0000 mHa (within chemical accuracy)
  Error vs CCSD          : +0.0002 mHa
  SQD subspace dim used  : 4  (shots=100000)
  <S^2>                  : 0.0000
  Excitation dist (|c|^2): HF 0.987  S 0.000  D 0.013  T 0.000  higher 0.000
  Circuit (ideal) : depth 45, 2q-gates 40, fits depth budget


## 6. Circuit analysis metrics
Expressibility (vs. Haar-random) and the excitation-rank decomposition (HF/S/D/T/Q/higher relative to Hartree-Fock) -- the same diagnostics used throughout this project to understand *what the policy is actually generating*, not just its energy. See `docs/2026-08-21/pipeline_overview_2026-08-21.pdf`'s "What recovery actually does to the sampled subspace" for a real 32-qubit example of this same histogram.

In [7]:
import ffsim
from sqd_gqe.gqe import circuit_metrics as cm

dim = ffsim.dim(system.norb, system.nelec)
state_fn = cm.make_state_fn_from_pool(pool, ngates=8)
print("expressibility", cm.expressibility_kl(state_fn, dim, n_states=60, seed=0))

ref = cm.ccsd_reference_state(system, pool)
ham = ffsim.MolecularHamiltonian(system.active_hamiltonian.one_body, system.active_hamiltonian.two_body, constant=float(system.active_hamiltonian.core_energy))
print("chemistry suitability", cm.chemistry_suitability(pool, list(rng.integers(0, len(pool), size=8)), reference_state=ref, hamiltonian=ham))

merged = pipeline.global_refined_state
if merged is not None and merged.strings is not None:
    w = np.abs(np.asarray(merged)) ** 2
    print("excitation-rank dist", cm.excitation_rank_histogram(merged.strings, system.nelec, weights=w))

expressibility {'expressibility_kl': 0.331609357499919, 'mean_fidelity': 0.3259499022081085, 'n_pairs': 1770}
chemistry suitability {'locality': 1.0, 'particle_number_ok': 1.0, 'spin_sq': 3.0814879110195774e-33, 'ref_overlap': 0.2673073753123059, 'energy': -0.2739411181303247}
excitation-rank dist {'HF': 0.9873338735229796, 'S': 0.0, 'D': 0.01266612647702047, 'T': 0.0, 'Q': 0.0, 'higher': 0.0, 'combined': 1.0, 'n_dets': 2}


## Next steps
- **Real IBM hardware** instead of `ffsim`: `tutorial/02_ibm_hardware.ipynb` -- same loop, one flag change (`--sampler-kind ibm_hardware`).
- **Multi-GPU SQD recovery** for larger molecules: `tutorial/03_environment_setup.md`.
- `ffsim`'s exact statevector simulation scales combinatorially with active-space size -- it becomes impractical well before 40 qubits. That is precisely the wall real hardware sampling is built to get around (see the architecture deck).